In [22]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
from torchvision import datasets, transforms

In [23]:
cancer = load_breast_cancer()
X = cancer.data
y = cancer.target

# 데이터 표준화
scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)
# stratify - 라벨이 불균형할 때 그 비율에 맞춰서 나눠라

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1) # 2진분류일 떄 shape 변경
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [24]:
class CancerClassifier(nn.Module):
    def __init__(self):
        super(CancerClassifier, self).__init__()
        self.fc1 = nn.Linear(30, 64) # 특성 30
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1) # 결과가 하나임

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.fc3(x)
        return x

In [25]:
model = CancerClassifier()
criterion = nn.BCEWithLogitsLoss() #시그모이드 + Binary Cross Entropy
optimizer = optim.Adam(model.parameters(), lr=0.001)

def train_model(epochs):
    model.train()
    for epoch in range(epochs):  # ✅ range로 수정
        for inputs, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()  # ✅ 누락된 backward() 추가
            optimizer.step()
        print(f"Epoch {epoch + 1} / {epochs}, Loss : {loss.item():.4f}")
    print("학습완료")

def evaluate_model():
    model.eval()  # ✅ .eval()로 수정
    with torch.no_grad():
        correct = 0
        total = 0
        for inputs, labels in test_loader:  # ✅ test_loader로 변경
            outputs = model(inputs)
            predicted = torch.round(torch.sigmoid(outputs))
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        accuracy = 100 * correct / total
        print(f"테스트 정확도: {accuracy:.2f}%")

In [26]:
if __name__ == "__main__":
    train_model(100)
    evaluate_model()

Epoch 1 / 100, Loss : 0.5023
Epoch 2 / 100, Loss : 0.4671
Epoch 3 / 100, Loss : 0.2958
Epoch 4 / 100, Loss : 0.1063
Epoch 5 / 100, Loss : 0.0925
Epoch 6 / 100, Loss : 0.0073
Epoch 7 / 100, Loss : 0.0069
Epoch 8 / 100, Loss : 0.0654
Epoch 9 / 100, Loss : 0.0350
Epoch 10 / 100, Loss : 0.0024
Epoch 11 / 100, Loss : 0.0010
Epoch 12 / 100, Loss : 0.0298
Epoch 13 / 100, Loss : 0.0834
Epoch 14 / 100, Loss : 0.0073
Epoch 15 / 100, Loss : 0.0305
Epoch 16 / 100, Loss : 0.0556
Epoch 17 / 100, Loss : 0.0001
Epoch 18 / 100, Loss : 0.0058
Epoch 19 / 100, Loss : 0.0041
Epoch 20 / 100, Loss : 0.0004
Epoch 21 / 100, Loss : 0.0098
Epoch 22 / 100, Loss : 0.0006
Epoch 23 / 100, Loss : 0.0004
Epoch 24 / 100, Loss : 0.0327
Epoch 25 / 100, Loss : 0.0002
Epoch 26 / 100, Loss : 0.0000
Epoch 27 / 100, Loss : 0.0004
Epoch 28 / 100, Loss : 0.0027
Epoch 29 / 100, Loss : 0.0520
Epoch 30 / 100, Loss : 0.0035
Epoch 31 / 100, Loss : 0.0043
Epoch 32 / 100, Loss : 0.0042
Epoch 33 / 100, Loss : 0.0145
Epoch 34 / 100, Los